# 原 Stage1 尾部概率与 SOXL 选择损失对齐

仅读取已经验证 NAS 的完整 Stage1 0bp Qlib Recorder。分组由已实现选择分量定义，只作事后归因；尾部头不是下一日 SOXL 相对损失标签，不从分组选择交易阈值。

In [1]:
import os,json,hashlib
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_original_tail_probability_loss_alignment_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda x:hashlib.sha256(Path(x).read_bytes()).hexdigest()
source=root/plan['sources']['signal_loss_card'];tail=root/plan['sources']['selection_tail_card']
assert sha(source/'summary.json')==plan['sources']['signal_loss_summary_sha256'] and sha(tail/'result.json')==plan['sources']['selection_tail_result_sha256']
src=json.loads((source/'summary.json').read_text());tail_result=json.loads((tail/'result.json').read_text())
assert src['recorder_id']==plan['sources']['signal_loss_recorder_id'] and src['nas_readback']['verified'] is True
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['sources']['Qlib_provider']),region='us',kernels=1)
with R.uri_context(src['recorder_uri']):
 rec=R.get_recorder(recorder_id=src['recorder_id'],experiment_id=src['experiment_id'])
 d=rec.load_object('analysis/daily')
assert len(d)==944 and d.trade_date.is_unique and d.signal_date.is_unique and np.isfinite(d.tail_probability).all()
assert abs(d.SOXL_selection_log.sum()-tail_result['full']['selection_log_sum'])<1e-10
q3=d.loc[d.trade_date.dt.to_period('Q').astype(str).eq('2026Q3')]
assert len(q3)==64 and abs(q3.SOXL_selection_log.sum()-tail_result['quarters']['2026Q3']['selection_log_sum'])<1e-10
worst=q3.nsmallest(2,'SOXL_selection_log').trade_date.dt.strftime('%Y-%m-%d').tolist()
assert set(worst)==set(plan['sources']['preidentified_trade_dates'])
def stats(frame):
 a=frame.tail_probability.astype(float)
 return {'n':len(frame),'mean':float(a.mean()) if len(a) else None,'median':float(a.median()) if len(a) else None,'q25':float(a.quantile(.25)) if len(a) else None,'q75':float(a.quantile(.75)) if len(a) else None}
def grouped(frame):
 return {'negative':stats(frame.loc[frame.SOXL_selection_log.lt(0)]),'positive':stats(frame.loc[frame.SOXL_selection_log.gt(0)]),'zero_contribution_days':int(frame.SOXL_selection_log.eq(0).sum()),'selection_log_sum':float(frame.SOXL_selection_log.sum())}
quarters=[]
for key,g in d.groupby(d.trade_date.dt.to_period('Q'),sort=True):quarters.append({'quarter':str(key),'sessions':len(g),**grouped(g)})
assert len(quarters)==16
fixed=[]
for day in plan['sources']['preidentified_trade_dates']:
 row=d.loc[d.trade_date.dt.strftime('%Y-%m-%d').eq(day)];assert len(row)==1
 x=row.iloc[0]
 fixed.append({'signal_date':str(pd.Timestamp(x.signal_date).date()),'trade_date':day,'tail_probability':float(x.tail_probability),'tail_probability_empirical_percentile_all_944':float(d.tail_probability.le(x.tail_probability).mean()),'alpha_probability':float(x.alpha_probability),'raw_nominal_beta':float(x.raw_nominal_beta),'own_state_beta_cap':float(x.own_state_beta_cap),'cap_binding':bool(x.cap_binding),'risk_state':int(x.risk_state),'open_execute':bool(x.open_execute),'SOXL_selection_log':float(x.SOXL_selection_log)})
result={'status':'original_Stage1_tail_head_predecision_probability_readback_all_quarters_descriptive_only','plan_sha256':sha(card/'plan.json'),'source_recorder_id':src['recorder_id'],'source_NAS_sha256':src['nas_evidence']['sha256'],'all_944':grouped(d),'quarters':quarters,'two_preidentified_loss_days':fixed,'tail_head_target_matches_one_day_SOXL_selection_label':False,'new_market_requests':0,'model_fits':0,'new_native_accounts':0,'bootstraps':0,'new_Recorders':0}
(card/'result.json').write_text(json.dumps(result,indent=2,ensure_ascii=False)+'\n')
print({'all_counts':(result['all_944']['negative']['n'],result['all_944']['positive']['n'],result['all_944']['zero_contribution_days']),'Q3_2026':next(x for x in quarters if x['quarter']=='2026Q3'),'fixed_days':fixed})

[15490:MainThread](2026-10-23 07:16:18,277) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[15490:MainThread](2026-10-23 07:16:18,279) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[15490:MainThread](2026-10-23 07:16:18,279) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{'all_counts': (336, 383, 225), 'Q3_2026': {'quarter': '2026Q3', 'sessions': 64, 'negative': {'n': 8, 'mean': 0.2572724837809801, 'median': 0.25719041377305984, 'q25': 0.2289087399840355, 'q75': 0.3027160242199898}, 'positive': {'n': 9, 'mean': 0.19156699048148262, 'median': 0.16556788980960846, 'q25': 0.1565493494272232, 'q75': 0.2353305071592331}, 'zero_contribution_days': 47, 'selection_log_sum': -0.14390082802460138}, 'fixed_days': [{'signal_date': '2026-07-02', 'trade_date': '2026-07-06', 'tail_probability': 0.3342364430427551, 'tail_probability_empirical_percentile_all_944': 0.9777542372881356, 'alpha_probability': 0.530587375164032, 'raw_nominal_beta': 1.3301900422546988, 'own_state_beta_cap': 1.3301900422546988, 'cap_binding': False, 'risk_state': 0, 'open_execute': True, 'SOXL_selection_log': -0.03904331168412108}, {'signal_date': '2026-07-24', 'trade_date': '2026-07-27', 'tail_probability': 0.3118877708911896, 'tail_probability_empirical_percentile_all_944': 0.965042372881356

## 中文结论

完整 944 日选择分量负/正/零为 336/383/225，负/正日尾部概率均值 0.13949/0.13832，未表现为稳定的次日 SOXL 相对选择区分器。2026Q3 两个预先锁定最大亏损持有日的前一信号日尾部概率为 0.33424/0.31189，虽位于本复用历史较高经验分位，但该分位不能成为交易阈值。原尾部头标签是 SOXX 五开盘路径最低值≤−6%，与本诊断单日选择符号不同。0 新拟合、账户或 bootstrap。逐季结果见 `result.json`，解释边界见 `conclusion.md`；SOXX 目标未达成。